# 01 · Explore TUSZ (P1)

Look at the audit tables and a few recordings before building the pipeline.

- Run `python scripts/00_audit_tusz.py` first (or `data.dry_run=true` without the dataset).
- Exploration only: anything reusable moves to `src/neuromech/`. Outputs are stripped on commit.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from neuromech.data import extract_ecg, extract_seizures, read_annotation, read_edf, to_tcp

DRY_RUN = True  # set False once TUSZ is available and the real audit has run
BASE = Path("../data/cache/dry_run") if DRY_RUN else Path("../data/processed")
recordings = pd.read_parquet(BASE / "metadata.parquet")
seizures = pd.read_parquet(BASE / "seizures.parquet")
recordings.head()

## Dataset size and coverage

In [ ]:
recordings.groupby("split").agg(
    recordings=("file", "size"),
    patients=("patient", "nunique"),
    hours=("duration_s", lambda s: s.sum() / 3600),
    ecg=("has_ecg", "mean"),
    seizures=("n_seizures", "sum"),
)

## Seizure types, durations and EEG available before onset

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
seizures["type"].value_counts().plot.bar(ax=axes[0], title="Seizure types")
seizures["duration_s"].plot.hist(bins=30, ax=axes[1], title="Seizure duration (s)")
(seizures["preictal_available_s"] / 60).plot.hist(bins=30, ax=axes[2], title="EEG before onset (min)")
plt.tight_layout()

## One recording in the TCP montage, with its seizures

In [ ]:
row = recordings[recordings["n_seizures"] > 0].iloc[0]
rec = read_edf(row["path"])
tcp = to_tcp(rec)
szs = extract_seizures(read_annotation(Path(row["path"]).with_suffix(".csv")))

t = np.arange(tcp.data.shape[1]) / tcp.fs
fig, ax = plt.subplots(figsize=(15, 8))
for i, ch in enumerate(tcp.channels):
    ax.plot(t, tcp.data[i] / 200 - i, lw=0.5, color="k")
for sz in szs:
    ax.axvspan(sz.onset, sz.offset, color="tab:red", alpha=0.15)
ax.set_yticks(-np.arange(len(tcp.channels)), tcp.channels)
ax.set_xlabel("time (s)")
ax.set_title(f"{row['file']}  ·  {[s.label for s in szs]}  ·  onset: {szs[0].onset_channels}")

## ECG (when present)

In [ ]:
ecg = extract_ecg(rec)
if ecg is None:
    print("no ECG in this recording")
else:
    x, fs = ecg
    plt.figure(figsize=(15, 2))
    plt.plot(np.arange(x.size) / fs, x, lw=0.5)
    plt.xlim(0, 20)
    plt.title("ECG, first 20 s")